# Codesign with Catapult

## List trained models

In [ ]:
#!for f in `ls weights/dataset_3src_16x16_weights/*/*.json`; do echo $f; done
!for f in `ls best-model/dataset_3src_16x16_50x12P5_centeredIncidence_weights/*/*.json`; do echo $f; done # CHANGE ME

## Setup

Please follow [these instructions](https://github.com/GiuseppeDiGuglielmo/catapult_venvs) to setup an environment for Catapult AI NN (_hls4ml_).

You can either use the _hls4ml_ release with Catapult (2024.2_1) or point to a Siemens or official GitHub repository. [Here we provide some details.](https://github.com/GiuseppeDiGuglielmo/catapult_compare_releases)

In [ ]:
# Path to the Catapult installation directory
!echo $MGC_HOME

In [ ]:
# Path to the hls4ml installation directory
# It can be point to either the Catapult installation directory or a copy of Siemens/Official GitHub repo 
!echo $PYTHONPATH

In [ ]:
# For non-quantized models: Conv2D_Max, Conv2D_Full, Conv2D_Slim, Conv1D_Full, Conv1D_Slim, Mlp_Full, Mlp_Slim
# For quantized models: QConv2D_Max, QConv2D_Full, QConv2D_Slim, QConv1D_Full, QConv1D_Slim, QMlp_Full, QMlp_Slim
model_type = 'QMlp_Slim' 

## Import libraries

Disable some console warnings on the ASIC-group servers

In [ ]:
import os

In [ ]:
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [ ]:
import sys
sys.path.insert(0, "/home/dajiang/smart-pixels-ml/two_bit_optimization_helpers") # Change this line to wherever the two_bit_optimization_helpers is located 

In [ ]:
import hls4ml

import numpy as np
import math
import tensorflow as tf
import yaml
import json
from two_bit_optimization_helpers import *

from matplotlib import colors
import matplotlib.pyplot as plt

from two_bit_optimization_helpers.loss import *
from qkeras import quantized_bits

from two_bit_optimization_helpers.codesign_utils import *

In [ ]:
!ls /nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/

## Set parameters

In [ ]:
dataset_base_dir = "/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/" # CHANGE ME

tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

npy_dir_val = os.path.join("npy")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train_contained") # CHANGE ME
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val_contained") # CHANGE ME

In [ ]:
# batch_size = 5000
# train_file_size = len(os.listdir(dataset_train_dir))
# val_file_size = len(os.listdir(dataset_validation_dir))

In [ ]:
base_dir = './best-model/dataset_3src_16x16_50x12P5_centeredIncidence_weights/' # CHANGE ME

fingerprint_choices = {
    # non-quantized models
    'Conv2D_Max'  : '392456de',
    'Conv2D_Full' : '392456de',
    'Conv2D_Slim' : '392456de',
    'Conv1D_Full' : '3eb13b90',
    'Conv1D_Slim' : '3eb13b90',
    'Mlp_Full'    : '392456de',
    'Mlp_Slim'    : '392456de',
    # quantized models
    'QConv2D_Max'  : '392456de',
    'QConv2D_Full' : '392456de',
    'QConv2D_Slim' : '392456de',
    'QConv1D_Full' : '3eb13b90',
    'QConv1D_Slim' : '3eb13b90',
    'QMlp_Full'    : '392456de',
    'QMlp_Slim'    : '392456de'
};

for k in fingerprint_choices.keys():
    hdf5_file = base_dir + f'weights-2t-{k}-2bit_optimized-{fingerprint_choices[k]}-checkpoints/best_model.hdf5'
    if(not os.path.exists(hdf5_file)):
        print(f"ERROR: {hdf5_file} does not exist")

fingerprint = fingerprint_choices[model_type]

In [ ]:
weights_dir = base_dir + f'weights-2t-{model_type}-2bit_optimized-{fingerprint}-checkpoints'
best_model_hdf5 = f"{weights_dir}/best_model.hdf5"
best_model_keras = f"{weights_dir}/best_model.keras"
best_model_weights_hdf5 = f"{weights_dir}/best_model_weights.hdf5"
best_model_weights_keras = f"{weights_dir}/best_model_weights.keras"
model_architecture_json = f"{weights_dir}/model_architecture.json"

In [ ]:
print(best_model_hdf5)

## Load data

In [ ]:
X = np.load(f'{npy_dir_val}/dataset_3sr_16x16_50x12P5_centeredIncidence_{model_type}_X_val_contained.npy')#[:batch_size,]
y = np.load(f'{npy_dir_val}/dataset_3sr_16x16_50x12P5_centeredIncidence_{model_type}_y_val_contained.npy')#[:batch_size,]

In [ ]:
print(X.shape)
print(y.shape)

### Visualize data

Data visualization can also be moved to a different notebook, which may also already exists.

In [ ]:
# Plot some events
for i in range(4):
    plot_event(X, range(2), i)

In [ ]:
ani = animate_event(X, range(X.shape[3]), 0)
ani.save("animation.gif", writer="imagemagick", fps=2)
HTML(ani.to_jshtml())

### Quantize data

<b style="background-color: yellow; color: red">ATTENTION: Input quantization is disabled. Data is already quantized.</i></b>

## QKeras model

### Load QKeras model

In [ ]:
# Load the whole model from HDF5 file
from tensorflow.keras.models import load_model
from qkeras.utils import _add_supported_quantized_objects

custom_loss_choices = {
    # non-quantized models
    'Conv2D_Max'  : {"custom_loss": custom_loss},
    'Conv2D_Full' : {"custom_diag_loss": custom_diag_loss},
    'Conv2D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'Conv1D_Full' : {"custom_diag_loss": custom_diag_loss},
    'Conv1D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'Mlp_Full'    : {"custom_diag_loss": custom_diag_loss},
    'Mlp_Slim'    : {"custom_sse_loss": custom_sse_loss},
    # quantized models
    'QConv2D_Max'  : {"custom_loss": custom_loss},
    'QConv2D_Full' : {"custom_diag_loss": custom_diag_loss},
    'QConv2D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'QConv1D_Full' : {"custom_diag_loss": custom_diag_loss},
    'QConv1D_Slim' : {"custom_sse_loss": custom_sse_loss},
    'QMlp_Full'    : {"custom_diag_loss": custom_diag_loss},
    'QMlp_Slim'    : {"custom_sse_loss": custom_sse_loss},
};

co = custom_loss_choices[model_type]
_add_supported_quantized_objects(co)
model = load_model(best_model_hdf5, custom_objects=co)
model.summary(line_length=120, show_trainable=True)

### Run QKeras model

In [ ]:
print(f'Input batch: {X.shape[0]}')
print(f'Input shape: {X.shape}')

#### QKeras Prediction

In [ ]:
y_qkeras = model.predict(np.ascontiguousarray(X))
print(f'QKeras output batch: {y_qkeras.shape[0]}')
print(f'QKeras output shape: {y_qkeras.shape}')

#### QKeras Profiling

In [ ]:
trace_qkeras = hls4ml.model.profiling.get_ymodel_keras(model, X)

In [ ]:
for key in trace_qkeras.keys():
    print(f'QKeras layer trace shape: {key} {trace_qkeras[key].shape}')

#### Save .dat files

In [ ]:
# Save input features and model predictions
np.savetxt(f"tb_input_features.dat", X.reshape(X.shape[0], -1), fmt="%f")
np.savetxt(f"tb_output_predictions.dat", y_qkeras.reshape(X.shape[0], -1), fmt="%f")

In [ ]:
!wc -l *tb_input_features.dat
!wc -l *tb_output_predictions.dat

## hls4ml model

### Configure hls4ml model

In [ ]:
project_name = f"SmtPxl_{model_type}"

strategy = "Latency"
io_type = "io_parallel"
programmable_weights = True

hls4ml_output_dir = f"{project_name}_{io_type}_{strategy}{'_pw' if programmable_weights else ''}_hls4ml_prj"

asiclibs = "tcbn28hpcplusbwp30p140ssg0p81v125c_ccs_dc"
asiclibspath ='/home/giuseppe/research/projects/smartpixels/tsmc28nm_lib'

In [ ]:
# Remove existing project
!rm -rf $hls4ml_output_dir*

In [ ]:
config_hls4ml = hls4ml.utils.config.create_config(
    backend = "Catapult",
    project_name = project_name,
    output_dir = hls4ml_output_dir,
    tech = "asic",
    asiclibs= asiclibs,
    asiclibspath = asiclibspath,
    asicfifo = "hls4ml_lib.mgc_pipe_mem",
    clock_period = 25,
    io_type = io_type,
    csim = False, SCVerify = False, Synth = True, BuildBUP = False
)

#print(json.dumps(config_hls4ml, indent=4))

In [ ]:
config_hls4ml["HLSConfig"] = hls4ml.utils.config_from_keras_model(
    model,
    granularity="name",
    default_precision="ac_fixed<16,6,true>",
    default_reuse_factor=1
)

config_hls4ml["HLSConfig"]["Model"]["Strategy"] = strategy

if programmable_weights:
    # All the weights are exposed through BRAMs
    config_hls4ml["HLSConfig"]["Model"]["BramFactor"] = 0

#print(json.dumps(config_hls4ml, indent=4))

<b style="background-color: yellow; color: red">ATTENTION: The model precision can be fine tuned here.</b>

#### Bit-precision

In [ ]:
print(model_type)

In [ ]:
match model_type:
    case 'Conv2D_Max' | 'Conv2D_Full' | 'Conv2D_Slim' | 'Conv1D_Full' | 'Conv1D_Slim' | 'Mlp_Full'  | 'Mlp_Slim' :
        # Kind of unsopported, we care about QKeras models instead, see below
        pass
    case 'QConv2D_Max'  :
        config_hls4ml["HLSConfig"]['LayerName']['input_1']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['q_dense_2']['Precision']['result'] = 'fixed<10,4>'
        config_hls4ml["HLSConfig"]['LayerName']['q_dense_2_linear']['Precision']['result'] = 'fixed<10,4>'
    case 'QConv2D_Full' :
        config_hls4ml["HLSConfig"]['LayerName']['input_1']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['q_dense_2']['Precision']['result'] = 'fixed<10,4>'
        config_hls4ml["HLSConfig"]['LayerName']['q_dense_2_linear']['Precision']['result'] = 'fixed<10,4>'
    case 'QConv2D_Slim' :
        config_hls4ml["HLSConfig"]['LayerName']['input_1']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['q_dense_2']['Precision']['result'] = 'fixed<10,4>'
        config_hls4ml["HLSConfig"]['LayerName']['q_dense_2_linear']['Precision']['result'] = 'fixed<10,4>'
    case 'QConv1D_Full' :
        config_hls4ml["HLSConfig"]['LayerName']['input_pxls']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['avg_pooling_2d_proj_x']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['avg_pooling_2d_proj_y']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3']['Precision']['result'] = 'fixed<10,4>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3_linear']['Precision']['result'] = 'fixed<10,4>'
    case 'QConv1D_Slim' :
        config_hls4ml["HLSConfig"]['LayerName']['input_pxls']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['avg_pooling_2d_proj_x']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['avg_pooling_2d_proj_y']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3']['Precision']['result'] = 'fixed<10,2>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3_linear']['Precision']['result'] = 'fixed<10,2>'
    case 'QMlp_Full'    :
        config_hls4ml["HLSConfig"]['LayerName']['input_pxls']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d_1']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3']['Precision']['result'] = 'fixed<10,4>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3_linear']['Precision']['result'] = 'fixed<10,4>'
    case 'QMlp_Slim'    :
        config_hls4ml["HLSConfig"]['LayerName']['input_pxls']['Precision']['result'] = 'ufixed<2,2>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['average_pooling2d_1']['Precision']['result'] = 'fixed<16,8>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3']['Precision']['result'] = 'fixed<10,4>'
        config_hls4ml["HLSConfig"]['LayerName']['dense_3_linear']['Precision']['result'] = 'fixed<10,4>'
    case _:
        ;    

In [ ]:
# Point to the model definition, weights/biase values and C++ testbench data files
config_hls4ml["KerasH5"] = best_model_weights_hdf5
config_hls4ml["KerasJson"] = model_architecture_json
config_hls4ml["InputData"] = f"tb_input_features.dat"
config_hls4ml["OutputPredictions"] = f"tb_output_predictions.dat"

#print(json.dumps(config_hls4ml, indent=4))

In [ ]:
# TODO: is this necessary?
with open(f"myproject_config.yml", "w") as yaml_file:
    yaml.dump(config_hls4ml, yaml_file, explicit_start=False, default_flow_style=False)

In [ ]:
# Enable tracing for all of the layers
for layer in config_hls4ml["HLSConfig"]["LayerName"].keys():
    print("Enable tracing for layer:", layer)
    config_hls4ml["HLSConfig"]["LayerName"][layer]["Trace"] = True

In [ ]:
# Convert QKeras model to Catapult HLS C++
hls_model_hls4ml = hls4ml.converters.keras_to_hls(config_hls4ml)

In [ ]:
hls4ml.utils.plot_model(hls_model_hls4ml, show_shapes=True, show_precision=True, to_file=f"qkeras.png")
hls4ml.utils.plot_model(hls_model_hls4ml, show_shapes=True, show_precision=True, to_file=None)

In [ ]:
# Writing HLS project
hls_model_hls4ml.compile()

### Run hls4ml model

#### hls4ml Prediction

In [ ]:
y_hls4ml = hls_model_hls4ml.predict(np.ascontiguousarray(X))

In [ ]:
print(f'hls4ml output batch: {y_hls4ml.shape[0]}')
print(f'hls4ml output shape: {y_hls4ml.shape}')

#### hls4ml Profiling

In [ ]:
# Run tracing on the test set for the hls4ml model (fixed-point precision) 
pred_hls4ml, trace_hls4ml = hls_model_hls4ml.trace(X)

## Compare QKeras and hls4ml

### Trace visual inspection

### MSE per layer

In [ ]:
print(model_type)

In [ ]:
def mse(actual, predicted):
    return ((actual - predicted) ** 2).mean()

# print(trace_hls4ml.keys())
# print(trace_hls4ml['q_separable_conv2d_pointwise'].shape)
# print(trace_qkeras['q_separable_conv2d'].shape)
 
for key in trace_hls4ml.keys():
    if key == "q_separable_conv2d_depthwise":
        continue
    print("-------")
    print("MSE {} {}".format(
        key.replace("_pointwise", ""),
        mse(
            trace_hls4ml[key].flatten(), 
            trace_qkeras[key.replace("_pointwise", "")].flatten())))

### Parity plots

We use [parity plots](https://en.wikipedia.org/wiki/Parity_plot) to compare _hls4ml_ and QKeras implementations of each layer.
- If the _hls4ml_ implementation is accurate (i.e. matches QKeras implementation), the ouput values should lie close to the diagonal line `y=x`, indicating equivalence.
- Deviations from this line highlight inaccuracies or implementation errors.

In [ ]:
plot_layer_comparisons(trace_hls4ml, trace_qkeras, save_path="comparison_qkeras_hls4ml.jpeg")

### Performance Variables

In [ ]:
print(model_type)

In [ ]:
import os
import pandas as pd

vars_dir = (
    "./smart-pixels-ml/processed_parquets/"
    "dataset_3src_16x16_50x12P5_centeredIncidence/"
    "test_dataset_3src_16x16_50x12P5_centeredIncidence/2bit_optimized"
)

def save_performance_variables(df: pd.DataFrame, model_type: str, fingerprint: str) -> None:
    os.makedirs(vars_dir, exist_ok=True)
    vars_file = "{}/2t-{}-2bit_optimized-{}-hls4ml-vars.parquet".format(
        vars_dir, model_type, fingerprint
    )
    print(vars_file)
    df.to_parquet(vars_file)

if model_type in ('QConv2D_Max',):
    # True values/Labels (QKeras)
    df = pd.DataFrame(
        y, 
        columns=[
            'xtrue',
            'ytrue',
            'cotAtrue',
            'cotBtrue',
        ]
    )

    # Predicted values (hls4ml)
    df['x']                = y_hls4ml[:, 0]
    df['M11']              = y_hls4ml[:, 1]
    df['y']                = y_hls4ml[:, 2]
    df['M22']              = y_hls4ml[:, 3]
    df['cotA']             = y_hls4ml[:, 4]
    df['M33']              = y_hls4ml[:, 5]
    df['cotB']             = y_hls4ml[:, 6]
    df['M44']              = y_hls4ml[:, 7]
    df['M21']              = y_hls4ml[:, 8]
    df['M31']              = y_hls4ml[:, 9]
    df['M32']              = y_hls4ml[:,10]
    df['M41']              = y_hls4ml[:,11]
    df['M42']              = y_hls4ml[:,12]
    df['M43']              = y_hls4ml[:,13]

    # Residuals (true - predicted) WITH PARENTHESES
    df['residuals_x']                 = df['xtrue']                - df['x']
    df['residuals_y']                 = df['ytrue']                - df['y']
    df['residuals_cotA']              = df['cotAtrue']             - df['cotA']
    df['residuals_cotB']              = df['cotBtrue']             - df['cotB']

    # Sigmas
    for m in ["M11","M22","M33","M44"]:
            df[m] = 1e-9 + tf.math.maximum(df[m], 0.0)
    
    df["sigmax"]           = abs(df["M11"])
    df["sigmay"]           = np.sqrt(df["M21"]**2 + df["M22"]**2)
    df["sigmacotA"]        = np.sqrt(df["M31"]**2 + df["M32"]**2 + df["M33"]**2)
    df["sigmacotB"]        = np.sqrt(df["M41"]**2 + df["M42"]**2 + df["M43"]**2 + df["M44"]**2)

    save_performance_variables(df, model_type, fingerprint)

elif model_type in ('QConv2D_Full', 'QConv1D_Full', 'QMlp_Full'):
    # True values/Labels (QKeras)
    df = pd.DataFrame(
        y,
        columns=[
            'xtrue',
            'ytrue',
            'cotAtrue',
            'cotBtrue'
        ]
    )

    # Predicted values (hls4ml)
    df['x']         = y_hls4ml[:, 0]
    df['y']         = y_hls4ml[:, 1]
    df['cotA']      = y_hls4ml[:, 2]
    df['cotB']      = y_hls4ml[:, 3]
    df['M11']       = y_hls4ml[:, 4]
    df['M22']       = y_hls4ml[:, 5]
    df['M33']       = y_hls4ml[:, 6]
    df['M44']       = y_hls4ml[:, 7]

    # Residuals
    df['residuals_x']         = df['xtrue']         - df['x']
    df['residuals_y']         = df['ytrue']         - df['y']
    df['residuals_cotA']      = df['cotAtrue']      - df['cotA']
    df['residuals_cotB']      = df['cotBtrue']      - df['cotB']

    # Sigmas
    mapping = {
            'M11': 'sigmax',
            'M22': 'sigmay',
            'M33': 'sigmacotA',
            'M44': 'sigmacotB',
        }
    for m, new_name in mapping.items():
        df[new_name] = tf.nn.softplus(df[m]) + 1e-9

    save_performance_variables(df, model_type, fingerprint)


elif model_type in ('QConv2D_Slim', 'QConv1D_Slim', 'QMlp_Slim'):
    # True values/Labels (QKeras)
    df = pd.DataFrame(
        y, 
        columns=[
            'xtrue',
            'ytrue',
            'cotBtrue'
        ]
    )

    # Predicted values (hls4ml)
    df['x']        = y_hls4ml[:, 0]
    df['y']        = y_hls4ml[:, 1]
    df['cotB']     = y_hls4ml[:, 2]

    # Residuals
    df['residuals_x']        = df['xtrue']    - df['x']
    df['residuals_y']        = df['ytrue']    - df['y']
    df['residuals_cotB']     = df['cotBtrue'] - df['cotB']

    save_performance_variables(df, model_type, fingerprint)

else:
    pass


## Model synthesis

In [ ]:
print(f'Catapult hls4ml project: {os.getcwd()}/{hls4ml_output_dir}')

<b style="background-color: yellow; color: red">ATTENTION: Run Catapult HLS from the command line</i></b>

```
cd SmtPxl_MODEL_TYPE_io_parallel_Latency_pw_hls4ml_prj
# edit build_prj.tcl
#    ...
#    Synth False
#    ...
catapult -f built_prj.tcl
# top/main loop pipeline II=1, unroll all of the other loops
```

### Results